In [1]:
import sys
sys.path[:0] = ['..']   # this repository: the extant package

import logging
import shutil
import warnings
from importlib import reload
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from matplotlib.gridspec import GridSpec
import numpy as np
import pandas as pd
import xarray as xr

from dask.distributed import wait
from extant import config
# from extant.plots import formatting, styles
from extant.utils import datatree
from extant.utils.logger import setup_logging

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)

#(t): Harmless warnings, silenced the same way in every notebook. zarr's, that parts of the stores (the text
#(t): coordinates, the consolidated metadata) aren't in the Zarr 3 standard yet:
warnings.filterwarnings('ignore', message='.*Zarr (format 3|V3) specification')
#(t): numpy's, about slices with nothing but NaN (land, a model without a member) or nothing at all, too few members for
#(t): a standard deviation, and dividing by zero or by NaN (the log of a probability of 0, say):
warnings.filterwarnings('ignore', message='All-NaN slice encountered', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Mean of empty slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Degrees of freedom <= 0 for slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='(divide by zero|invalid value) encountered', category=RuntimeWarning)
#(t): dask's, about large task graphs, which are large on purpose:
warnings.filterwarnings('ignore', message='.*Sending large graph of size', category=UserWarning)
#(t): Every figure's text at the notebook's sizes and colours (styles.NOTEBOOK), as in every notebook
# plt.rcParams.update(styles.NOTEBOOK.rc)
from importlib.util import find_spec

#(t): flox speeds up groupby/resample on dask (the seasonal means); xarray uses it automatically when installed
if find_spec('flox') is None:
    logger.warning('flox is not installed: pip install flox to speed up the seasonal means')

In [2]:
import src.paths as paths

In [3]:
VARIABLE = config.VARIABLES['tas']

In [4]:
#(t): Start a Dask Gateway cluster, or reuse the one already running, and send it the extant package
#(c): Re-run upload_package after editing anything in extant/: the workers run its functions
from extant.utils import jasmin

gateway, cluster, client = jasmin.connect(n_workers=20, wait_for=6)
jasmin.upload_package(client)
client

GatewayClusterError: Cluster 'fefea7b6e432402c9e96e55e159f0733' failed to start, see logs for more information

In [ ]:
import src.paths as paths
import dask.dataframe as ddf
from dask import delayed, compute

In [ ]:
paths.OBS_PATH

In [ ]:
list(paths.OBS_PATH.iterdir())

# LESFMIP

In [ ]:
#(t): Where notebook calc_01's seasonal means are, and where this notebook's results go: JASMIN's scratch; and the sample in
#(t): this repository
SCRATCH_DIR = paths.SCRATCH_DIR
SAMPLE_DIR = Path('../tests/data')
SCRATCH_DIR, SAMPLE_DIR

In [ ]:
#(t): The seasonal store notebook calc_01 saved
seasonal_path = SCRATCH_DIR / 'seasonal' / f'lesfmip_{VARIABLE.name}.zarr'
#(t): Only the experiments analysed (filter keeps the model nodes above them)
#(t): Opened lazily
lesfmip_season_tree = xr.open_datatree(seasonal_path, engine='zarr', chunks={})

In [ ]:
lesfmip_season_tree

# ERA5

In [ ]:
era5_path = paths.SCRATCH_DIR / 'seasonal' / f'era5_{VARIABLE.name}.zarr'
era5_path

In [ ]:
#(t): ERA5's seasonal means
era5_seasonal_da = xr.open_zarr(era5_path)[VARIABLE.name].load()

In [ ]:
era5_seasonal_da

# Station Data

## Opening Data

In [ ]:
meta_data = pd.read_csv('../docs/LookUpTable_Observations_Metadata_v12082026.csv', index_col='Station')

In [ ]:
meta_data

In [ ]:
list(Path('/gws/ssde/j25a/extant/users/rywill/Extremes_Database').iterdir())

In [ ]:
list((paths.OBS_PATH / 'AWS'/ 'Amery' / 'LGB69').iterdir())

In [ ]:
all_paths = np.array(list(paths.OBS_PATH.rglob('*/**/*_T.csv')))
all_paths[:3]

In [ ]:
@delayed
def _open_obs_date(path):
    df = pd.read_csv(path, usecols=['Date', 'air_temperature (Cel)'],
                     index_col='Date', parse_dates=True)
    
    return df.rename(columns={'air_temperature (Cel)': path.stem}).sort_index()

In [ ]:
all_station_df = [_open_obs_date(path) for path in all_paths]
all_station_df = compute(*[_open_obs_date(p) for p in all_paths])
station_df = pd.concat(all_station_df, axis=1, join='outer', sort=True)
station_df.index.is_monotonic_increasing

In [ ]:
station_ds = station_df.rename_axis(index='time', columns='station').stack().to_xarray()
station_ds

In [ ]:
import re

coords = {'Latitude (degrees_north)': 'lat', 'Longitude (degrees_east)': 'lon'}

# longest names first, so substrings don't match early
names = sorted(meta_data.index, key=len, reverse=True)
pattern = '(' + '|'.join(map(re.escape, names)) + ')'

# match each station to its metadata name
meta_name = station_ds.station.to_index().str.extract(pattern, expand=False)

# look up lat/lon in station order
md = meta_data.reindex(meta_name)[list(coords)].rename(columns=coords)

station_ds = station_ds.assign_coords(
    meta_name=('station', meta_name),
    lat=('station', md['lat'].to_numpy()),
    lon=('station', md['lon'].to_numpy()),
)
station_ds

In [ ]:
station_ds.isel(station=slice(0, 10)).plot(hue='station', add_legend=False);

## Seasonal Conversion

In [ ]:
plt.plot(station_ds.time.values)

In [ ]:
station_season_ds = station_ds.resample(time='QS-DEC').mean()
station_season_valid_ds = station_ds.resample(time='QS-DEC').count()
# Convert to percent
station_season_valid_ds = 100*station_season_valid_ds/xr.ones_like(station_ds).resample(time='QS-DEC').sum()
# Remvoe all with percent less than 90
station_season_ds = station_season_ds.where(station_season_valid_ds > 90)

In [ ]:
station_season_ds

In [ ]:
def split_segments(da, gap_years=10):
    """Split each station into separate series wherever there's a gap > gap_years."""
    valid = da.notnull()
    t = da.time.where(valid)
    prev = t.ffill('time').shift(time=1)
    new_seg = (t - prev) > np.timedelta64(int(gap_years * 365.25), 'D')
    seg = new_seg.cumsum('time').where(valid)

    nseg = seg.max('time') + 1
    split = nseg.station[nseg > 1].values
    print(f'{len(split)} station(s) split: {list(split)}')

    parts = []
    for k in range(int(seg.max()) + 1):
        p = da.where(seg == k).dropna('station', how='all')
        p = p.assign_coords(station=[f'{s}_seg_{k+1:02d}' for s in p.station.values])
        parts.append(p)

    return xr.concat(parts, dim='station').sortby('station')



In [ ]:
def _pct_valid(x):
    ok = ~np.isnan(x)
    if not ok.any():
        return np.nan, 0
    idx = np.flatnonzero(ok)
    return 100 * ok[idx[0]:idx[-1] + 1].mean(), ok.sum()

In [ ]:
station_season_ds = split_segments(station_season_ds, gap_years=10)

In [ ]:
pct_valid, n_valid = xr.apply_ufunc(
    _pct_valid, station_season_ds,
    input_core_dims=[['time']],
    output_core_dims=[[], []],
    vectorize=True,
)

In [ ]:
keep = (n_valid > 40) & (pct_valid > 80)
station_robust_ds = station_season_ds.sel(station=keep)

In [ ]:
station_robust_ds

In [ ]:
station_robust_ds.sel(station='Station_Islands_Signy_T_seg_01').plot()
station_robust_ds.sel(station='Station_Islands_Signy_T_seg_02').plot()

In [ ]:
plt.bar(n_valid.station.values, n_valid.values)

In [ ]:
plt.bar(pct_valid.station.values, pct_valid.values)

In [ ]:
from extant.utils import time_axis

In [ ]:
time_axis.split_time

In [ ]:
station_ds = time_axis.split_time(station_robust_ds)

In [ ]:
above_15_years = station_ds.notnull().sum(dim='year') > 15
valid = above_15_years.sum(dim='season') == 4
valid

In [ ]:
station_ds = station_ds.sel(station=valid)

In [ ]:
station_ds

In [ ]:
station_ds.notnull().sum(dim='year').plot(row='season', figsize=(12, 6))

In [ ]:
station_season_valid_ds.isel(station=0).plot()
station_ds.isel(station=0).plot()
station_season_ds_t.isel(station=0).plot(color='purple', linestyle='--')

In [ ]:
station_ds.isel(station=slice(0, 10)).plot(hue='season', col='station', col_wrap=3);

## Trends

In [ ]:
from extant.statistics import trend

In [ ]:
station_linear_ds = xr.Dataset({'fit': trend.linear_fit(station_ds), 'gradient': trend.trend(station_ds)})

In [ ]:
station_linear_ds

In [ ]:
station_ds.isel(station=0, season=0).plot()
station_linear_ds.fit.isel(station=0, season=0).plot()

In [ ]:
station_linear_ds.gradient.plot()

# Merging ERA5 and stations

In [ ]:
def add_era5(ds):
    """Adds ERA5 at the grid point nearest the branch's station, on the branch's months, as product 'era5'."""
    era5_da = era5_time_da.sel(lat=ds.lat.item(), lon=ds.lon.item(), method='nearest').drop_vars(['lat', 'lon'])
    era5_da = era5_da.reindex(time=ds.time)
    return ds.merge(era5_da.expand_dims(product=['era5']).to_dataset(name='tas'))

In [ ]:
station_ds.isel(station=10).year.notnull()

In [ ]:
def trim(obj, time_coord='year'):
    valid = obj.notnull()
    if isinstance(obj, xr.Dataset):
        valid = valid.to_array().any('variable')  # valid if any variable is non-null
    other = [d for d in valid.dims if d != time_coord]
    if other:
        valid = valid.any(other)
    t = obj[time_coord].where(valid, drop=True)
    if t.size == 0:
        return obj.isel({time_coord: slice(0, 0)})
    return obj.sel({time_coord: slice(t[0].item(), t[-1].item())})


In [ ]:
def keep_coords(ds, keep=('lat', 'lon')):
    """Drop scalar coords except those in keep."""
    return ds.drop_vars([c for c in ds.coords if c not in ds.dims and c not in keep])

In [ ]:
from functools import reduce

In [ ]:
def nearest_point(station, gridded, coords=('lat', 'lon'), name=None):
    """Gridded data (DataArray, Dataset or DataTree) at the grid point nearest the station."""
    loc = {c: station[c].item() for c in coords}

    def pick(ds):
        if not all(c in ds.dims for c in coords):      # empty group nodes
            return ds
        out = ds.sel(loc, method='nearest').assign_coords(loc)
        return out.rename({v: name for v in out.data_vars}) if name else out

    if isinstance(gridded, xr.DataTree):
        return gridded.map_over_datasets(pick)
    if isinstance(gridded, xr.DataArray):
        gridded = gridded.to_dataset(name=name or gridded.name)
    return pick(gridded)


def match_node(node, how='overlap', match=('year', 'season'), keep_full=('year',),
               ref='station', nan=False):
    """Align every leaf under one station node (works with nested children like lesfmip/model/exp).

    how='match'   -> everything reindexed to ref's match coords
    how='overlap' -> everything trimmed to the match coords all leaves share
    how='full'    -> like overlap, but dims in keep_full (e.g. year) left untouched
    nan=True      -> NaN wherever any leaf is NaN (same gaps in all)
    """
    datasets = {leaf.relative_to(node): leaf.to_dataset() for leaf in node.leaves if leaf.data_vars}
    keys = [ref] + [k for k in datasets if k != ref]
    join = {'match': 'left', 'overlap': 'inner', 'full': 'inner'}[how]
    align_dims = set(match) - set(keep_full) if how == 'full' else set(match)
    all_dims = set().union(*(datasets[k].dims for k in keys))
    aligned = xr.align(*(datasets[k] for k in keys), join=join, exclude=all_dims - align_dims)

    if nan:
        def valid(ds):
            v = ds.to_array().notnull()
            return v.any([d for d in v.dims if d not in match])
        masks = xr.align(*(valid(ds) for ds in aligned), join='outer', fill_value=False)
        mask = reduce(lambda a, b: a & b, masks)
        aligned = [ds.where(mask.reindex_like(ds, fill_value=False)) for ds in aligned]

    return xr.DataTree.from_dict(dict(zip(keys, aligned)))

In [ ]:
station_tree = xr.DataTree.from_dict({
    str(s): keep_coords(trim(station_ds.sel(station=s)).to_dataset(name='tas'))
    for s in station_ds.station.values
})

In [ ]:
# datatree.streamline_tree(lesfmip_season_tree.match('*/historical'))

In [ ]:
station_collection_tree = xr.DataTree.from_dict({
    key: xr.DataTree.from_dict({
        'station': node.to_dataset(),
        'era5': nearest_point(node, era5_seasonal_da, name='tas'),
        # 'lesfmip': nearest_point(node, datatree.streamline_tree(lesfmip_season_tree.match('*/historical')), name='tas'),

    })
    for key, node in station_tree.children.items()
})
station_matched_tree = match_tree(station_collection_tree, how='overlap', nan=True)

In [ ]:
def match_to_station(ds):
    if 'tas' not in ds:
        return ds, ds
    obs, mod = xr.align(station_ds, ds.tas, join='inner', exclude={'member'})
    mask = (obs.notnull().reset_coords(drop=True)
            & mod.notnull().any('member').reset_coords(drop=True))   # present in both
    return obs.where(mask).to_dataset(name='tas'), mod.where(mask).to_dataset(name='tas')

lesfmip_obs, lesfmip_model = lesfmip_stn.map_over_datasets(match_to_station)

In [ ]:
pts = dict(lat=station_ds.lat, lon=station_ds.lon)
hist = datatree.streamline_tree(lesfmip_season_tree.match('*/historical'))

lesfmip_stn = xr.DataTree.from_dict({
    m: node.to_dataset().sel(pts, method='nearest').assign_coords(model=m)
    for m, node in hist.children.items()
}).compute()

In [ ]:
lesfmip_obs, lesfmip_model = lesfmip_stn.map_over_datasets(match_to_station)

In [ ]:
# station_era5_ds = datatree.tree_to_dataset(station_matched_tree, dims=['station', 'source']).tas
station_era5_ds = datatree.tree_to_dataset(
    station_matched_tree.filter(lambda n: 'lesfmip' not in n.path),
    dims=['station', 'source']).tas

In [ ]:
station_era5_ds.isel(station=0).plot(col='season', hue='source', col_wrap=2)

In [ ]:
station_era5_ds

# Trends

In [ ]:
from importlib import reload
from extant.statistics import trend

In [ ]:
reload(trend)

In [ ]:
lesfmip_matched

In [ ]:
lesfmip_obs_trend = lesfmip_obs.map_over_datasets(datatree.skip_empty(trend.trend), kwargs=dict(dim='year'))
lesfmip_model_trend = lesfmip_model.map_over_datasets(datatree.skip_empty(trend.trend), kwargs=dict(dim='year'))
lesfmip_obs_fit = lesfmip_obs.map_over_datasets(datatree.skip_empty(trend.linear_fit), kwargs=dict(dim='year'))
lesfmip_model_fit = lesfmip_model.map_over_datasets(datatree.skip_empty(trend.linear_fit), kwargs=dict(dim='year'))

In [ ]:
def detrended_std(data, fit):
    if 'tas' not in data:
        return data
    resid = data.tas - fit.tas
    return resid.std('year').to_dataset(name='std')

lesfmip_obs_std = lesfmip_obs.map_over_datasets(datatree.skip_empty(detrended_std), lesfmip_obs_fit)
lesfmip_model_std = lesfmip_model.map_over_datasets(datatree.skip_empty(detrended_std), lesfmip_model_fit)

In [ ]:
lesfmip_std_within = lesfmip_obs_std.map_over_datasets(datatree.skip_empty(in_range), lesfmip_model_std)

In [ ]:
def in_range(obs, model):
    if 'tas' not in obs:
        return obs
    o, m = obs.tas, model.tas
    return ((m.min('member') <= o) & (o <= m.max('member'))).to_dataset(name='within')

In [ ]:
# datatree.skip_empty

In [ ]:
lesfmip_within = lesfmip_obs_trend.map_over_datasets(datatree.skip_empty(in_range), lesfmip_model_trend)

In [ ]:
# model_da = lesfmip_model_trend['CESM2'].tas
# obs_da = lesfmip_obs_trend['CESM2'].tas

In [ ]:
# within = (model_da.min('member') <= obs_da) & (obs_da <= model_da.max('member'))

In [ ]:
# lesfmip_matched_trend = lesfmip_matched.map_over_datasets(
#     datatree.skip_empty(trend.trend), kwargs=dict(dim='year'))

In [ ]:
lesfmip_matched_trend

In [ ]:
lesfmip_matched_trend_ds = datatree.tree_to_dataset(lesfmip_matched_trend.mean(dim='member'), dims=['model'])#.squeeze('variable')
lesfmip_matched_trend_ds

In [ ]:
station_match_trend_ds = xr.Dataset({'fit': trend.linear_fit(station_era5_ds, 'year'),
                                     'gradient': trend.trend(station_era5_ds, 'year')})

In [ ]:
station_match_mk_ds = trend.grid_mk(station_era5_ds, 'year')

In [ ]:
station_match_mk_ds

In [ ]:
station_match_trend_ds.gradient.isel(season=0, source=0)

In [ ]:
import seaborn as sns

In [ ]:
station_era5_ds.sel(station='AWS_Amery_Relay_Station_T_seg_01').plot(hue='source', col='season')

In [ ]:
station_match_trend_ds.sel(station='AWS_Amery_Relay_Station_T_seg_01').fit.plot(hue='source', col='season')

In [ ]:
lesfmip_matched_trend_ds

In [ ]:
from matplotlib.colors import ListedColormap
tidy = lambda x: x.assign_coords(station=x.station.str.replace('AWS_', '').str.replace('_T_seg_01', ''))
def load(tree, var):
    da = datatree.tree_to_dataset(tree, dims=['model'])[var]
    return tidy(da).reset_coords(drop=True)      # drop lat/lon so obs and model don't clash

def position(o, m):
    """0 = obs below member range, 1 = within, 2 = above."""
    pos = xr.where(o < m.min('member'), 0, xr.where(o > m.max('member'), 2, 1))
    return pos.where(o.notnull())

trend_pos = position(load(lesfmip_obs_trend, 'tas'), load(lesfmip_model_trend, 'tas'))
std_pos = position(load(lesfmip_obs_std, 'std'), load(lesfmip_model_std, 'std'))
cat = (trend_pos * 3 + std_pos).transpose('season', 'station', 'model')

from matplotlib.colors import to_rgb

#           (trend, std): 0 = below, 2 = above
corners = {(0, 0): '#2166ac', (0, 2): '#1b9e77', (2, 0): '#7b3294', (2, 2): '#d7191c'}

def colour(t, s):
    if (t, s) in corners:
        return to_rgb(corners[(t, s)])
    if (t, s) == (1, 1):
        return (1, 1, 1)                                             # within both -> white
    near = [to_rgb(c) for (ct, cs), c in corners.items() if abs(ct - t) + abs(cs - s) == 1]
    return tuple(0.5 * np.mean(near, axis=0) + 0.5)                 # mix of neighbours, paled

cmap = ListedColormap([colour(t, s) for t in range(3) for s in range(3)])

seasons = cat.season.values
fig, axes = plt.subplots(1, len(seasons), sharey=True,
                         figsize=(4 * len(seasons) + 2, 0.3 * cat.sizes['station'] + 2))
for ax, season in zip(axes, seasons):
    c = cat.sel(season=season)
    ax.imshow(c.values, cmap=cmap, vmin=-0.5, vmax=8.5, aspect='auto', interpolation='none')
    ax.set_xticks(range(c.sizes['model']), c.model.values, rotation=90)
    ax.set_yticks(range(c.sizes['station']), c.station.values)
    ax.set_title(season)

# 2D legend
lax = fig.add_axes([0.95, 0.4, 0.07, 0.2])
lax.imshow(np.arange(9).reshape(3, 3), cmap=cmap, vmin=-0.5, vmax=8.5, origin='lower')
lax.set_xticks(range(3), ['below', 'within', 'above'], rotation=90)
lax.set_yticks(range(3), ['below', 'within', 'above'])
lax.set_xlabel('obs std')
lax.set_ylabel('obs trend')

In [ ]:
plt.rcParams['hatch.color'] = 'white'
cols = {0: '#d6604d', 1: '#f0f0f0', 2: '#4393c3'}     # pos 0 = obs below -> model over
labels = {0: 'model over', 1: 'model within', 2: 'model under'}

# record length = years with any station data; order stations longest first
record = tidy(station_ds.notnull().any('season').sum('year')).reset_coords(drop=True)
order = record.sortby(record, ascending=False).station.values
record, tp, sp = record.sel(station=order), trend_pos.sel(station=order), std_pos.sel(station=order)

seasons = ['DJF', 'MAM', 'JJA', 'SON']
y = np.arange(len(order))
ratios = [1, 0.25] + sum([[1, 1, 0.25] for _ in seasons], [])[:-1]   # record, gap, trend, std, gap, ...
fig, grid = plt.subplots(1, len(ratios), sharey=True, gridspec_kw={'width_ratios': ratios, 'wspace': 0.05},
                         figsize=(2.6 * len(seasons) + 2, 0.22 * len(y) + 1.5))

# record length
grid[0].barh(y, record.values, 0.8, color='#888')
grid[0].set_title('record\n(years)', fontsize=9)
grid[0].spines[['top', 'right', 'left']].set_visible(False)
grid[0].tick_params(left=False, labelsize=7)
grid[1].axis('off')

for s, season in enumerate(seasons):
    for p, (pos, name) in enumerate([(tp, 'trend'), (sp, 'std')]):
        ax = grid[2 + 3 * s + p]
        left = np.zeros(len(y))
        for k in [2, 1, 0]:                                      # under | within | over
            n = (pos.sel(season=season) == k).sum('model').values
            ax.barh(y, n, 0.8, left=left, color=cols[k], edgecolor='white', lw=0.3)
            left += n
        ax.set_title(f'{season}\n{name}' if p == 0 else f'\n{name}', fontsize=9)
        ax.set_xlim(0, cat.sizes['model'])
        ax.spines[['top', 'right', 'left']].set_visible(False)
        ax.tick_params(left=False, labelsize=7)
for ax in grid[4::3]:
    ax.axis('off')                                               # spacer columns between seasons

grid[0].set_yticks(y, order, fontsize=7)
grid[0].invert_yaxis()
fig.legend(handles=[Patch(color=cols[k], label=labels[k]) for k in [0, 1, 2]],
           loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.02), frameon=False)

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.path as mpath
from matplotlib.colors import BoundaryNorm

pos, season = trend_pos, 'DJF'        # swap to std_pos / another season

pct = {k: ((pos.sel(season=season) == k).sum('model')
           / pos.sel(season=season).notnull().sum('model') * 100) for k in range(3)}
record = tidy(station_ds.notnull().any('season').sum('year')).reset_coords(drop=True)
loc = tidy(xr.Dataset(dict(lat=station_ds.lat, lon=station_ds.lon)))
order = record.sortby(record, ascending=False).station.values   # big points first, small on top
scale = 2.5                                                       # marker area per year of record

# cmap = sns.color_palette('mako_r', as_cmap=True)
cmap = sns.color_palette('BrBG', as_cmap=True)

bounds = np.arange(0, 110, 10)
norm = BoundaryNorm(bounds, cmap.N)

theta = np.linspace(0, 2 * np.pi, 100)
circle = mpath.Path(np.vstack([np.sin(theta), np.cos(theta)]).T * 0.5 + 0.5)

panels = [(2, 'model under'), (1, 'model within'), (0, 'model over')]
fig, axes = plt.subplots(1, 3, figsize=(16, 6.5), subplot_kw=dict(projection=ccrs.SouthPolarStereo()),
                         gridspec_kw={'wspace': 0.02})
for ax, (k, title) in zip(axes, panels):
    ax.set_extent([-180, 180, -90, -58], ccrs.PlateCarree())
    ax.set_boundary(circle, transform=ax.transAxes)
    ax.add_feature(cfeature.LAND, color='#ececec', zorder=0)
    ax.coastlines(lw=0.4, color='#999')
    sc = ax.scatter(loc.lon.sel(station=order), loc.lat.sel(station=order),
                    c=pct[k].sel(station=order), s=record.sel(station=order) * scale,
                    cmap=cmap, norm=norm, edgecolor='#333', lw=0.4,
                    transform=ccrs.PlateCarree(), zorder=3)
    ax.set_title(f'{title} ({season})')

fig.colorbar(sc, ax=axes, orientation='horizontal', shrink=0.35, pad=0.04,
             ticks=bounds, label='% of models')

handles, lbls = sc.legend_elements('sizes', num=4, func=lambda s: s / scale, color='grey')
axes[-1].legend(handles, lbls, title='record (yrs)', loc='upper left', bbox_to_anchor=(1, 1), frameon=False)

In [ ]:
pos = trend_pos                       # swap to std_pos

seasons = ['DJF', 'MAM', 'JJA', 'SON']
pct = (pos == 1).sum('model') / pos.notnull().sum('model') * 100      # % within, (station, season)
record = tidy(station_ds.notnull().any('season').sum('year')).reset_coords(drop=True)
loc = tidy(xr.Dataset(dict(lat=station_ds.lat, lon=station_ds.lon)))
order = record.sortby(record, ascending=False).station.values
scale = 4

cmap = sns.color_palette('BrBG', as_cmap=True)
bounds = np.arange(0, 110, 10)
norm = BoundaryNorm(bounds, cmap.N)

theta = np.linspace(0, 2 * np.pi, 100)
circle = mpath.Path(np.vstack([np.sin(theta), np.cos(theta)]).T * 0.5 + 0.5)

fig, axes = plt.subplots(2, 2, figsize=(13, 13), subplot_kw=dict(projection=ccrs.SouthPolarStereo()),
                         gridspec_kw={'wspace': 0.02, 'hspace': 0.08})
for ax, season in zip(axes.flat, seasons):
    ax.set_extent([-180, 180, -90, -58], ccrs.PlateCarree())
    ax.set_boundary(circle, transform=ax.transAxes)
    ax.add_feature(cfeature.LAND, color='#ececec', zorder=0)
    ax.coastlines(lw=0.4, color='#999')
    sc = ax.scatter(loc.lon.sel(station=order), loc.lat.sel(station=order),
                    c=pct.sel(station=order, season=season), s=record.sel(station=order) * scale,
                    cmap=cmap, norm=norm, edgecolor='#333', lw=0.4,
                    transform=ccrs.PlateCarree(), zorder=3)
    ax.set_title(season, fontsize=14)

fig.colorbar(sc, ax=axes, orientation='horizontal', shrink=0.4, pad=0.03,
             ticks=bounds, label='% of models within')

handles, lbls = sc.legend_elements('sizes', num=4, func=lambda s: s / scale, color='grey')
axes[0, 1].legend(handles, lbls, title='record (yrs)', loc='upper left', bbox_to_anchor=(1, 1), frameon=False)

In [ ]:
seasons = ['DJF', 'MAM', 'JJA', 'SON']
pct_trend = (trend_pos == 1).sum('model') / trend_pos.notnull().sum('model') * 100
pct_std = (std_pos == 1).sum('model') / std_pos.notnull().sum('model') * 100
record = tidy(station_ds.notnull().any('season').sum('year')).reset_coords(drop=True)
loc = tidy(xr.Dataset(dict(lat=station_ds.lat, lon=station_ds.lon)))
order = record.sortby(record, ascending=False).station.values
scale = 3

cmap = sns.color_palette('BrBG', as_cmap=True)
bounds = np.arange(0, 110, 10)
norm = BoundaryNorm(bounds, cmap.N)

theta = np.linspace(0, 2 * np.pi, 100)
circle = mpath.Path(np.vstack([np.sin(theta), np.cos(theta)]).T * 0.5 + 0.5)

fig, axes = plt.subplots(2, 4, figsize=(22, 11.5), subplot_kw=dict(projection=ccrs.SouthPolarStereo()),
                         gridspec_kw={'wspace': 0.02, 'hspace': 0.05})
for row, (pct, name) in enumerate([(pct_trend, 'trend'), (pct_std, 'std')]):
    for col, season in enumerate(seasons):
        ax = axes[row, col]
        ax.set_extent([-180, 180, -90, -58], ccrs.PlateCarree())
        ax.set_boundary(circle, transform=ax.transAxes)
        ax.add_feature(cfeature.LAND, color='#ececec', zorder=0)
        ax.coastlines(lw=0.4, color='#999')
        sc = ax.scatter(loc.lon.sel(station=order), loc.lat.sel(station=order),
                        c=pct.sel(station=order, season=season), s=record.sel(station=order) * scale,
                        cmap=cmap, norm=norm, edgecolor='#333', lw=0.4,
                        transform=ccrs.PlateCarree(), zorder=3)
        if row == 0:
            ax.set_title(season, fontsize=14)
        if col == 0:
            ax.text(-0.05, 0.5, name, transform=ax.transAxes, rotation=90,
                    ha='right', va='center', fontsize=14)

fig.colorbar(sc, ax=axes, orientation='horizontal', shrink=0.3, pad=0.03,
             ticks=bounds, label='% of models within')

handles, lbls = sc.legend_elements('sizes', num=4, func=lambda s: s / scale, color='grey')
axes[0, -1].legend(handles, lbls, title='record (yrs)', loc='upper left', bbox_to_anchor=(1, 1), frameon=False)

In [ ]:
models = cat.model.values
ncols = 6
nrows = int(np.ceil(len(models) / ncols))
fig, axes = plt.subplots(nrows, ncols, sharey=True,
                         figsize=(1.6 * ncols + 2, nrows * (0.2 * cat.sizes['station'] + 1.5)))

for ax, model in zip(axes.flat, models):
    c = cat.sel(model=model).transpose('station', 'season')
    ax.imshow(c.values, cmap=cmap, vmin=-0.5, vmax=8.5, aspect='auto', interpolation='none')
    ax.set_xticks(range(c.sizes['season']), c.season.values, rotation=90)
    ax.set_yticks(range(c.sizes['station']), c.station.values, fontsize=6)
    ax.set_title(model, fontsize=9)
for ax in axes.flat[len(models):]:
    ax.axis('off')                                   # hide unused panels

# 2D legend
lax = fig.add_axes([0.95, 0.4, 0.07, 0.15])
lax.imshow(np.arange(9).reshape(3, 3), cmap=cmap, vmin=-0.5, vmax=8.5, origin='lower')
lax.set_xticks(range(3), ['below', 'within', 'above'], rotation=90)
lax.set_yticks(range(3), ['below', 'within', 'above'])
lax.set_xlabel('obs std')
lax.set_ylabel('obs trend')

In [ ]:
tidy = lambda x: x.assign_coords(station=x.station.str.replace('AWS_', '').str.replace('_T_seg_01', ''))
# trend.trend is per year, so x10 for per decade
obs_per_decade = tidy(datatree.tree_to_dataset(lesfmip_obs_trend, dims=['model']).tas.mean('model')) * 10
mod_per_decade = tidy(datatree.tree_to_dataset(lesfmip_model_trend, dims=['model']).tas.mean('member')) * 10
within = tidy(datatree.tree_to_dataset(lesfmip_within, dims=['model']).within)
vmax = np.nanpercentile(np.abs(mod_per_decade.values), 95)

seasons = obs_per_decade.season.values
fig, axes = plt.subplots(1, len(seasons), sharey=True,
                         figsize=(5 * len(seasons), 0.3 * obs_per_decade.sizes['station'] + 2))
for ax, season in zip(axes, seasons):
    df = mod_per_decade.sel(season=season).transpose('station', 'model').to_pandas()
    df.insert(0, ' ', np.nan)                                  # gap column
    df.insert(0, 'station', obs_per_decade.sel(season=season).to_pandas())
    sns.heatmap(df, cmap="RdBu_r", center=0, vmin=-vmax, vmax=vmax,
                cbar=season == seasons[-1], cbar_kws={"label": "trend per decade"}, ax=ax)

    # outline model cells where obs is within the member range (+2 skips station + gap columns)
    w = within.sel(season=season).transpose('station', 'model').to_pandas().reindex(columns=df.columns[2:])
    for i, j in np.argwhere(w.fillna(False).values.astype(bool)):
        ax.add_patch(Rectangle((j + 2, i), 1, 1, fill=False, ec="k", lw=1))

    ax.set_title(season)
    ax.set_xlabel('')
    if ax is not axes[0]:
        ax.set_ylabel('')

In [ ]:
from matplotlib.patches import Rectangle

# trend.trend is per year, so x10 for per decade
trend_per_decade = (station_match_trend_ds.gradient.transpose('station', 'source', 'season')
         .pipe(lambda x: x.assign_coords(station=x.station.str.replace('AWS_', '').str.replace('_T_seg_01', '')))) * 10
sig = station_match_mk_ds.sig.transpose('station', 'source', 'season')
vmax = np.nanpercentile(np.abs(trend_per_decade.values), 95)

seasons = trend_per_decade.season.values
fig, axes = plt.subplots(1, len(seasons), sharey=True,
                         figsize=(2.2 * len(seasons) + 1, 0.35 * trend_per_decade.sizes['station'] + 1))
for ax, season in zip(axes, seasons):
    t, s = trend_per_decade.sel(season=season), sig.sel(season=season)
    sns.heatmap(t.to_pandas(), annot=True, fmt="+.2f", cmap="RdBu_r", center=0,
                vmin=-vmax, vmax=vmax, annot_kws={"fontsize": 7},
                cbar=season == seasons[-1], cbar_kws={"label": "trend per decade"}, ax=ax)
    # outline significant cells; sig is NaN with under 10 years, which argwhere would count as True
    for i, j in np.argwhere((s == 1).values):
        ax.add_patch(Rectangle((j, i), 1, 1, fill=False, ec="k", lw=1.5))
    ax.set_title(season)
    ax.set_xlabel('')
    if ax is not axes[0]:
        ax.set_ylabel('')

In [ ]:
fig, ax = plt.subplots(1, 1)
station_match_trend_ds.fit.isel(station=0, season=0).plot(ax=ax, hue='source')
station_era5_ds.isel(station=0, season=0).plot(ax=ax, hue='source')

# Old (Using Pandas)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# # datetime index, drop all-NaN columns
# station_df.index = pd.to_datetime(df.index)
# station_df = df.dropna(axis=1, how="all")

def completeness(s):
    s = s.loc[s.first_valid_index():s.last_valid_index()]  # trim leading/trailing NaNs
    return pd.Series({
        "start": s.index[0],
        "end": s.index[-1],
        "length": s.index[-1] - s.index[0],
        "n": len(s),
        "pct_complete": 100 * s.notna().mean(),
    })

summary = station_df.apply(completeness).T
summary

In [ ]:


# availability plot: one row per station, sorted by start
fig, ax = plt.subplots(figsize=(12, 8))

order = summary.sort_values("start").index
for y, col in enumerate(order):
    ax.plot(station_df.index, station_df[col].notna().map({True: y, False: np.nan}), lw=3)

ax.set_yticks(range(len(order)), order)
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

In [ ]:
# monthly mean, masked where month <= 95% complete
station_df.index = pd.to_datetime(station_df.index)
frac = station_df.notna().resample("MS").mean()
obs_monthly = station_df.resample("MS").mean().where(frac > 0.95)
obs_monthly

In [ ]:
obs_monthly.shape

In [ ]:
obs_monthly.plot(legend=False)

In [ ]:
# clip to model period end
obs_monthly = obs_monthly.loc[:"2014"]

In [ ]:
SEASON_GROUPS = {'DJF': [12, 1, 2], 'MMA': [3,4,5], 'JJA': [6,7,8], 'SON': [9, 10, 11]}

In [ ]:
# obs_monthly.iloc[obs_monthly.index.month.isin([12, 1, 2]), :]

In [ ]:
obs_monthly.head()

In [ ]:
# station_dse.index.month.isin#.iloc[station_dse.index.month.isin(month_list), :]

In [ ]:
obs_monthly.to_xarray()

In [ ]:
# split each station at NaN gaps, keep continuous runs >= 15 years
min_len = 15 * 12
segments = {}
for col in obs_monthly:
    station_dse = obs_monthly[col]
    for season, month_list in SEASON_GROUPS.items():
        station_dse = station_dse.iloc[station_dse.index.month.isin(month_list)]
        valid = station_dse.notna()
        run_id = (valid != valid.shift()).cumsum()
        runs = [seg for _, seg in station_dse[valid].groupby(run_id[valid]) if len(seg) >= min_len]
        for i, seg in enumerate(runs):
            segments[f"{col}_{i}_{season}"] = seg
obs_segments = pd.DataFrame(segments)


In [ ]:
obs_segments.tail()

In [ ]:
obs_segments.shape

In [ ]:
# monthly anomalies (remove each segment's seasonal cycle)
obs_segments.index = pd.to_datetime(obs_segments.index)

In [ ]:
obs_segments_ds = xr.DataArray(obs_segments.rename_axis(index="time", columns="station"), name="tas")
obs_segments_ds

In [ ]:
import extant.statistics.trend as trend

In [ ]:
obs_season_ds = obs_segments_ds.resample(time='QS-DEC').mean()

In [ ]:
obs_trend_fit = trend.linear_fit(obs_season_ds, dim='time')

In [ ]:

obs_anom = obs_segments - obs_segments.groupby(obs_segments.index.month).transform("mean")
obs_anom

In [ ]:
trend

In [ ]:
obs_season_ds

In [ ]:
# linear fit vs decimal years: slope (/yr), intercept, p-value
from scipy.stats import linregress

def panda_trend(s):
    s = s.dropna()
    t = s.index.year + (s.index.month - 0.5) / 12
    r = linregress(t, s.values)
    return pd.Series({"slope": r.slope, "intercept": r.intercept, "pvalue": r.pvalue})

In [ ]:
# trends table, one row per segment, sorted by start
order = obs_anom.apply(lambda s: s.first_valid_index()).sort_values().index
trends = obs_anom[order].apply(panda_trend).T
trends["slope_dec"] = trends["slope"] * 10
trends

In [ ]:
trends.index.name = 'Name'

In [ ]:
trends.index.values

In [ ]:
import re

coords = {'Latitude (degrees_north)':'lat', 'Longitude (degrees_east)':'lon'}

# Get all the names form teh meta data - these are subsets of the other names
names = sorted(meta_data.index.values, key=len, reverse=True)

pattern = '(' + '|'.join(map(re.escape,names)) + ')'

station = trends.index.to_series().str.extract(pattern, expand=False)

relevant_metadata = (
    meta_data.reindex(station)[list(coords)]
    .set_axis(trends.index)
    .assign(station=station.values)
    .rename(columns=coords)
)

relevant_metadata.head()

In [ ]:
trends = pd.merge(trends, relevant_metadata, left_index=True, right_index=True)

In [ ]:
lat, lon = 'Latitude (degrees_north)', 'Longitude (degrees_east)'

station_ds = (
    trends
    # .rename(columns={lat: 'lat', lon: 'lon'})
    .rename_axis('station')
    .to_xarray()
    .set_coords(['lat', 'lon'])
)

In [ ]:
era5_station_match = era5_seasonal_da.sel(
    lat=station_ds.lat, 
    lon=station_ds.lon%360,
    method='nearest'
)

In [ ]:
obs_anom

In [ ]:
obs_season_ds

In [ ]:
obs_trend_fit

In [ ]:
from scipy.stats import kendalltau

In [ ]:
# panel plot: anomalies, 12-month mean, trend (red if p<0.05)
fig, axes = plt.subplots(len(order), 1, figsize=(10, 0.8 * len(order) + 1), sharex=True)

obs_grads = {}
era_grads = {}

for ax, station in zip(np.atleast_1d(axes), obs_trend_fit.station.values):
    smet = relevant_metadata.loc[station, :]
    da = obs_season_ds.sel(station=station).dropna('time')
    da_era = era5_time_da.sel(lat=smet.lat, lon=smet.lon, method='nearest').dropna('time')

    # common overlap at month resolution (works for cftime and datetime64)
    ym = lambda d, i: d.time[i].dt.strftime('%Y-%m').item()
    t0 = max(ym(da, 0), ym(da_era, 0))
    t1 = min(ym(da, -1), ym(da_era, -1))
    
    da = da.sel(time=slice(t0, t1))
    da_era = da_era.sel(time=slice(t0, t1))

    ax.axhline(0, color="0.85", lw=0.5)
    ax.plot(da.time.values, da.values, color="0.7", lw=0.5)
    
    ax.plot(da_era.time.values, da_era.values)

    da_fit = trend.linear_fit(da, 'time')
    era_fit = trend.linear_fit(da_era, 'time')
    p_da = kendalltau(da.time.values, da.values).pvalue
    p_era = kendalltau(da_era.time.values, da_era.values).pvalue
    
    ax.plot(da_fit.time.values, da_fit.values,
            color="C3" if p_da < 0.05 else "0.5", lw=1.5)
        
    ax.plot(era_fit.time.values, era_fit.values,
            color="C3" if p_era < 0.05 else "0.5", lw=1.5)

    obs_grads[station] = trend.trend(da, 'time').item() * 10
    era_grads[station] = trend.trend(da_era, 'time').item() * 10

#     ax.text(0, 0.9, f"{col.replace('Station_', '')}  {tr.slope_dec:+.2f}/dec",
#             transform=ax.transAxes, fontsize=8, va="top")
#     ax.yaxis.set_major_locator(plt.MaxNLocator(2))
#     ax.tick_params(labelsize=7)
#     for side in ["top", "right"]:
#         ax.spines[side].set_visible(False)

# axes[-1].xaxis.set_major_locator(mdates.YearLocator(10))
# axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
# fig.supylabel("Temperature anomaly")
# fig.tight_layout(h_pad=0.2)

In [ ]:
tend_obs_model = pd.DataFrame({'obs':obs_grads, 'era':era_grads})

In [ ]:
tend_obs_model.head()

In [ ]:
vals = tend_obs_model.values
vmax = np.nanmax(np.abs(vals))
fig, ax = plt.subplots(figsize=(3, 0.35 * len(tend_obs_model) + 1))

im = ax.imshow(vals, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")

# axis labels from the dataframe
ax.set_xticks(range(vals.shape[1]), tend_obs_model.columns, rotation=45, ha="right")
ax.set_yticks(range(vals.shape[0]), tend_obs_model.index)

# annotate each cell
for (i, j), v in np.ndenumerate(vals):
    ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7)

fig.colorbar(im, ax=ax, label="K/decade")

In [ ]:
# panel plot: anomalies, 12-month mean, trend (red if p<0.05)
fig, axes = plt.subplots(len(order), 1, figsize=(10, 0.55 * len(order) + 1), sharex=True)

for ax, col in zip(np.atleast_1d(axes), order):
    anom = obs_anom[col].dropna()
    tr = trends.loc[col]
    t = anom.index.year + (anom.index.month - 0.5) / 12

    ax.axhline(0, color="0.85", lw=0.5)
    ax.plot(anom.index, anom, color="0.7", lw=0.5)
    ax.plot(anom.index, anom.rolling(12, center=True).mean(), color="k", lw=1)
    ax.plot(anom.index, tr.intercept + tr.slope * t,
            color="C3" if tr.pvalue < 0.05 else "0.5", lw=1.5)

    ax.text(0, 0.9, f"{col.replace('Station_', '')}  {tr.slope_dec:+.2f}/dec",
            transform=ax.transAxes, fontsize=8, va="top")
    ax.yaxis.set_major_locator(plt.MaxNLocator(2))
    ax.tick_params(labelsize=7)
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)

axes[-1].xaxis.set_major_locator(mdates.YearLocator(10))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
fig.supylabel("Temperature anomaly")
fig.tight_layout(h_pad=0.2)

In [ ]:
# heatmap of trend per decade; outlined if p<0.05
from matplotlib.patches import Rectangle

vals = trends[["slope_dec"]].values
vmax = np.nanmax(np.abs(vals))
fig, ax = plt.subplots(figsize=(3, 0.35 * len(trends) + 1))

im = ax.imshow(vals, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")

for i, (val, p) in enumerate(zip(trends.slope_dec, trends.pvalue)):
    ax.text(0, i, f"{val:+.2f}", ha="center", va="center", fontsize=8)
    if p < 0.05:
        ax.add_patch(Rectangle((-0.5, i - 0.5), 1, 1, fill=False, ec="k", lw=2))

ax.set_yticks(range(len(trends)), trends.index.str.replace("Station_", ""), fontsize=8)
ax.set_xticks([])
fig.colorbar(im, ax=ax, label="Trend (per decade)")
fig.tight_layout()

In [ ]:
# trends = obs_segments.apply(trend)
# trends

# Old

In [ ]:
obs_metadata = {}

for df in all_station_df:
    start, end = np.take(df.index.values, [0, -1])
    valid = np.isfinite(df.values.squeeze())
    length = len(valid)
    valid = valid[valid]
    obs_metadata[df.columns[0]] = {
        'start': pd.to_datetime(start), 'end': pd.to_datetime(end), 'n_points':length, 'valid': len(valid), 'percent': 100*len(valid)/length}

In [ ]:
obs_metadata = pd.DataFrame(obs_metadata).T

In [ ]:
obs_metadata['percent'].plot()

In [ ]:
obs_metadata.shape[0]

In [ ]:
obs_metadata.query('percent > 95').shape[0]

In [ ]:
obs_metadata.sort_values('start')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

order = summary.sort_values("start").index
for y, col in enumerate(order):
    ax.plot(df.index, df[col].notna().map({True: y, False: np.nan}), lw=3)

ax.set_yticks(range(len(order)), order)
ax.xaxis.set_major_locator(mdates.YearLocator(5))

In [ ]:
import matplotlib.dates as mdates

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

y_change = 0.05
y = 0
for label, row in obs_metadata.sort_values('start').iterrows():
    y += y_change
    ax.plot([pd.to_datetime(row.start), pd.to_datetime(row.end)], [y, y])

ax.xaxis.set_major_locator(mdates.YearLocator(5))

In [ ]:
obs_metadata.loc[:, 'percent'].plot()

In [ ]:
obs_metadata.loc[:, 'valid'].plot()

In [ ]:
station_df = all_station_df[0].join(all_station_df[1:30], how='outer').compute()

In [ ]:
station_df.shape

In [ ]:
station_df.plot()

In [ ]:
all_paths.shape

In [ ]:
import dask.dataframe as ddf

In [ ]:
df = ddf.read_csv('/gws/ssde/j25a/extant/users/rywill/Extremes_Database/Obs/AWS/Amery/LGB69/AWS_Amery_LGB69_T.csv')

In [ ]:
df = df[['Date',  'air_temperature (Cel)']].compute().set_index('Date')

In [ ]:
df.head()

In [ ]:
df.plot()